# M03 — Thí nghiệm 2: Khảo sát Tối ưu Không gian màu & Dung hợp Đặc trưng

**Mã nhiệm vụ:** `M03`

**Người thực hiện:** Nguyên

**Mục tiêu:** Kiểm chứng Giả thuyết H2 — so sánh 3 bộ đặc trưng: `HOG Gray`, `HOG YUV` (HOG trên 3 kênh Y/U/V nối lại), và `HOG Gray + Color Hist (HSV)` (dung hợp hình học + màu sắc), nhằm chứng minh sự kết hợp đa thức là tối ưu.

Toàn bộ pipeline dùng đặc trưng thủ công (HOG + Color Histogram) + SVM (scikit-learn).

**Notebook này chạy TRỰC TIẾP cấu hình tiền xử lý chiến thắng ở M02** (`outputs/M02_handover_config.json`) — tuyệt đối không tự ý đổi size/mode để đảm bảo tính nhất quán.

> Chạy notebook `M02_experiment_1_preprocessing_and_size.ipynb` trước và đảm bảo file `outputs/M02_handover_config.json` đã được tạo (cùng thư mục làm việc với notebook này) trước khi chạy notebook này.

In [1]:
import os
import cv2
import json
import time
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Đã import xong toàn bộ thư viện cần thiết.")


Đã import xong toàn bộ thư viện cần thiết.


## 1. Bước 1 — Tiếp nhận cấu hình chiến thắng từ M02

Đọc file `outputs/M02_handover_config.json` , lấy  `winner_size`, `winner_mode` để tiếp tục.

In [2]:
HANDOVER_PATH = "outputs/M02_handover_config.json"

if not os.path.exists(HANDOVER_PATH):
    raise FileNotFoundError(
        f"Không tìm thấy {HANDOVER_PATH}.\n"
        f"-> Hãy chạy notebook M02_experiment_1_preprocessing_and_size.ipynb TRƯỚC "
        f"để sinh ra file cấu hình bàn giao này."
    )

with open(HANDOVER_PATH, "r", encoding="utf-8") as f:
    handover_config = json.load(f)

WINNER_SIZE = handover_config["winner_size"]
WINNER_MODE = handover_config["winner_mode"]
DATASET_DIR = handover_config["dataset_dir"]
TRAIN_SUBDIR = handover_config["train_subdir"]
VAL_SUBDIR = handover_config["val_subdir"]
CLASSES = handover_config["classes"]

TRAIN_DIR = os.path.join(DATASET_DIR, TRAIN_SUBDIR)
VAL_DIR = os.path.join(DATASET_DIR, VAL_SUBDIR)

print("Đã nhận cấu hình từ M02:")
print(f"  Size = {WINNER_SIZE}x{WINNER_SIZE}")
print(f"  Mode = {WINNER_MODE}")
print(f"  Dataset dir = {DATASET_DIR}")
print(f"  Số lớp = {len(CLASSES)}")


Đã nhận cấu hình từ M02:
  Size = 48x48
  Mode = stretch
  Dataset dir = E:\Nguyen\AIL303m_FUDN_SUM26\paper\cropped_dataset
  Số lớp = 47


In [3]:
# Nếu DATASET_DIR trong file cấu hình không đúng máy hiện tại, sửa lại ở đây:
# DATASET_DIR = r"E:\Nguyen\AIL303m_FUDN_SUM26\paper\cropped_dataset"
# TRAIN_DIR = os.path.join(DATASET_DIR, TRAIN_SUBDIR)
# VAL_DIR = os.path.join(DATASET_DIR, VAL_SUBDIR)

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".ppm")

def list_split(split_dir):
    # Trả về (samples, classes) cho một thư mục train/val dạng ImageFolder.
    # Sắp xếp cả lớp lẫn tên file để thứ tự đọc CỐ ĐỊNH -> kết quả tái lập được
    # (đồng bộ với M02).
    if not os.path.isdir(split_dir):
        raise FileNotFoundError(f"Không tìm thấy thư mục: {split_dir}")
    samples = []
    classes = sorted([d for d in os.listdir(split_dir)
                       if os.path.isdir(os.path.join(split_dir, d))])
    for cls in classes:
        cls_dir = os.path.join(split_dir, cls)
        for fname in sorted(os.listdir(cls_dir)):
            if fname.lower().endswith(IMG_EXTS):
                samples.append((os.path.join(cls_dir, fname), cls))
    return samples, classes

train_samples, train_classes = list_split(TRAIN_DIR)
val_samples, val_classes = list_split(VAL_DIR)

assert train_classes == val_classes, "Danh sách lớp giữa train và val không khớp nhau!"

print(f"Số ảnh train: {len(train_samples)}")
print(f"Số ảnh val:   {len(val_samples)}")


Số ảnh train: 6605
Số ảnh val:   824


## 2. Bước 1 — Hàm resize kế thừa từ M02

In [4]:
# Hàm resize kế thừa NGUYÊN VẸN từ M02.
# Nội suy chọn theo hướng scale để không làm lệch đặc trưng giữa 2 notebook:
#   - Thu nhỏ (ảnh gốc lớn hơn đích)  -> INTER_AREA
#   - Phóng to (ảnh gốc nhỏ hơn đích) -> INTER_LINEAR
def _pick_interpolation(src_h, src_w, target_size):
    return cv2.INTER_AREA if max(src_h, src_w) >= target_size else cv2.INTER_LINEAR

def resize_stretch(img, target_size):
    h, w = img.shape[:2]
    interp = _pick_interpolation(h, w, target_size)
    return cv2.resize(img, (target_size, target_size), interpolation=interp)

def resize_pad_square(img, target_size):
    h, w = img.shape[:2]
    L = max(h, w)
    scale = target_size / L
    new_w, new_h = max(1, round(w * scale)), max(1, round(h * scale))
    interp = _pick_interpolation(h, w, target_size)
    resized = cv2.resize(img, (new_w, new_h), interpolation=interp)

    pad_w = target_size - new_w
    pad_h = target_size - new_h
    top = pad_h // 2
    bottom = pad_h - top
    left = pad_w // 2
    right = pad_w - left

    border_value = [0, 0, 0] if len(img.shape) == 3 else 0
    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right,
        borderType=cv2.BORDER_CONSTANT, value=border_value
    )
    # Sau khi đệm, ảnh LUÔN đúng target_size x target_size -> không resize lại (tránh mờ thừa).
    assert padded.shape[:2] == (target_size, target_size), \
        f"Kích thước sau pad sai: {padded.shape[:2]} != {(target_size, target_size)}"
    return padded

RESIZE_FUNCS = {"stretch": resize_stretch, "pad_square": resize_pad_square}
resize_fn = RESIZE_FUNCS[WINNER_MODE]

print(f"Sẽ resize toàn bộ ảnh về {WINNER_SIZE}x{WINNER_SIZE} bằng chế độ '{WINNER_MODE}'.")


Sẽ resize toàn bộ ảnh về 48x48 bằng chế độ 'stretch'.


## 3. Bước 2 — 3 hàm trích xuất đặc trưng

1. **`HOG Gray`**: HOG chuẩn trên ảnh xám.
2. **`HOG YUV`**: chuyển sang không gian màu YUV, trích HOG riêng trên từng kênh Y, U, V rồi nối (`np.concatenate`).
3. **`HOG Gray + Color Hist (HSV)`**: HOG Gray nối với Color Histogram 3D trên HSV (`bins=(8,8,8)`, chuẩn hoá xác suất), ghép bằng `np.hstack`.

In [5]:
HOG_PARAMS = dict(orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), block_norm="L2-Hys")
# Với size 32x32, dùng pixels_per_cell nhỏ hơn để đủ số block (kế thừa logic từ M02)
if WINNER_SIZE <= 32:
    HOG_PARAMS["pixels_per_cell"] = (4, 4)

print("Tham số HOG dùng cho toàn bộ M03:", HOG_PARAMS)


def extract_hog_gray(img_resized):
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY) if len(img_resized.shape) == 3 else img_resized
    return hog(gray, feature_vector=True, **HOG_PARAMS)


def extract_hog_yuv(img_resized):
    # Chuyển ảnh sang YUV, trích HOG riêng trên từng kênh Y, U, V rồi nối lại.
    yuv = cv2.cvtColor(img_resized, cv2.COLOR_BGR2YUV)
    feats = []
    for ch in range(3):
        channel = yuv[:, :, ch]
        feats.append(hog(channel, feature_vector=True, **HOG_PARAMS))
    return np.concatenate(feats)


def extract_color_hist_hsv(img_resized, bins=(8, 8, 8)):
    # Color Histogram 3D trên không gian màu HSV, chuẩn hoá thành phân phối xác suất.
    hsv = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, bins,
                         [0, 180, 0, 256, 0, 256])
    hist = hist.flatten()
    hist = hist / (hist.sum() + 1e-8)
    return hist


def extract_hog_gray_plus_hist(img_resized):
    hog_feat = extract_hog_gray(img_resized)
    hist_feat = extract_color_hist_hsv(img_resized)
    return np.hstack([hog_feat, hist_feat])


FEATURE_EXTRACTORS = {
    "HOG Gray": extract_hog_gray,
    "HOG YUV": extract_hog_yuv,
    "HOG Gray + Color Hist (HSV)": extract_hog_gray_plus_hist,
}

print("Đã định nghĩa xong 3 bộ trích xuất đặc trưng.")


Tham số HOG dùng cho toàn bộ M03: {'orientations': 9, 'pixels_per_cell': (8, 8), 'cells_per_block': (2, 2), 'block_norm': 'L2-Hys'}
Đã định nghĩa xong 3 bộ trích xuất đặc trưng.


In [6]:
def imread_unicode(path):
    # cv2.imread KHÔNG đọc được đường dẫn chứa ký tự Unicode (dấu tiếng Việt) trên Windows.
    # np.fromfile mở file bằng Python (hỗ trợ Unicode) rồi cv2.imdecode giải mã trong bộ nhớ.
    try:
        data = np.fromfile(path, dtype=np.uint8)
    except (OSError, ValueError):
        return None
    if data.size == 0:
        return None
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

def build_feature_matrix(samples, extractor_fn):
    X, y = [], []
    skipped = 0
    for filepath, label in samples:
        img = imread_unicode(filepath)
        if img is None:
            skipped += 1
            continue
        img_resized = resize_fn(img, WINNER_SIZE)
        feat = extractor_fn(img_resized)
        X.append(feat)
        y.append(label)
    if skipped:
        print(f"    [CẢNH BÁO] Bỏ qua {skipped}/{len(samples)} ảnh không đọc được.")
    return np.array(X, dtype=np.float32), np.array(y)

print("Đã định nghĩa xong build_feature_matrix() (đọc ảnh hỗ trợ đường dẫn Unicode).")


Đã định nghĩa xong build_feature_matrix() (đọc ảnh hỗ trợ đường dẫn Unicode).


## 4. Bước 2 — Ma trận thí nghiệm: 3 lần chạy

| Mã TN | Bộ đặc trưng |
|---|---|
| TN2.1 | `HOG Gray` |
| TN2.2 | `HOG YUV` |
| TN2.3 | `HOG Gray + Color Hist (HSV)` |

In [7]:
EXPERIMENTS_M03 = [
    {"code": "TN2.1", "name": "HOG Gray"},
    {"code": "TN2.2", "name": "HOG YUV"},
    {"code": "TN2.3", "name": "HOG Gray + Color Hist (HSV)"},
]

results_m03 = []
feature_cache = {}  # lưu lại X_train, X_val, y_train, y_val của từng cấu hình để bàn giao

for exp in EXPERIMENTS_M03:
    code_name = exp["code"]
    feat_name = exp["name"]
    extractor_fn = FEATURE_EXTRACTORS[feat_name]
    print(f"--- Đang chạy {code_name}: {feat_name} ---")

    X_train, y_train = build_feature_matrix(train_samples, extractor_fn)
    X_val, y_val = build_feature_matrix(val_samples, extractor_fn)

    n_features = X_train.shape[1]

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=10.0, gamma="scale", class_weight="balanced",
                     random_state=RANDOM_STATE)),
    ])

    t0 = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred = clf.predict(X_val)
    acc = accuracy_score(y_val, y_pred) * 100
    f1_macro = f1_score(y_val, y_pred, average="macro") * 100

    results_m03.append({
        "Mã TN": code_name,
        "Bộ đặc trưng": feat_name,
        "Số chiều (n_features)": n_features,
        "Val Accuracy (%)": round(acc, 2),
        "Val Macro F1 (%)": round(f1_macro, 2),
        "Thời gian Train (s)": round(train_time, 2),
    })

    feature_cache[feat_name] = {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
    }

    print(f"    -> Val Acc={acc:.2f}%  Val F1={f1_macro:.2f}%  n_features={n_features}  train_time={train_time:.2f}s")

df_results_m03 = pd.DataFrame(results_m03)
df_results_m03


--- Đang chạy TN2.1: HOG Gray ---
    -> Val Acc=96.48%  Val F1=95.95%  n_features=900  train_time=5.06s
--- Đang chạy TN2.2: HOG YUV ---
    -> Val Acc=96.36%  Val F1=94.75%  n_features=2700  train_time=21.30s
--- Đang chạy TN2.3: HOG Gray + Color Hist (HSV) ---
    -> Val Acc=95.87%  Val F1=95.19%  n_features=1412  train_time=7.38s


,Mã TN,Bộ đặc trưng,Số chiều (n_features),Val Accuracy (%),Val Macro F1 (%),Thời gian Train (s)
0,TN2.1,HOG Gray,900,96.48,95.95,5.06
1,TN2.2,HOG YUV,2700,96.36,94.75,21.30
2,TN2.3,HOG Gray + Color Hist (HSV),1412,95.87,95.19,7.38


In [8]:
os.makedirs("outputs", exist_ok=True)
df_results_m03.to_csv("outputs/M03_results_3_configs.csv", index=False, encoding="utf-8-sig")
print("Đã lưu: outputs/M03_results_3_configs.csv")
df_results_m03


Đã lưu: outputs/M03_results_3_configs.csv


,Mã TN,Bộ đặc trưng,Số chiều (n_features),Val Accuracy (%),Val Macro F1 (%),Thời gian Train (s)
0,TN2.1,HOG Gray,900,96.48,95.95,5.06
1,TN2.2,HOG YUV,2700,96.36,94.75,21.30
2,TN2.3,HOG Gray + Color Hist (HSV),1412,95.87,95.19,7.38


## 5. Bước 3 — Phân tích & Kết luận

In [9]:
gray_row = df_results_m03[df_results_m03["Bộ đặc trưng"] == "HOG Gray"].iloc[0]
yuv_row = df_results_m03[df_results_m03["Bộ đặc trưng"] == "HOG YUV"].iloc[0]
fusion_row = df_results_m03[df_results_m03["Bộ đặc trưng"] == "HOG Gray + Color Hist (HSV)"].iloc[0]

print("So sánh số chiều đặc trưng:")
print(f"  HOG Gray                    : {gray_row['Số chiều (n_features)']} chiều")
print(f"  HOG YUV                     : {yuv_row['Số chiều (n_features)']} chiều "
      f"(gấp {yuv_row['Số chiều (n_features)'] / gray_row['Số chiều (n_features)']:.1f} lần HOG Gray)")
print(f"  HOG Gray + Color Hist (HSV) : {fusion_row['Số chiều (n_features)']} chiều")
print()

print("So sánh Val Macro F1:")
print(f"  HOG Gray                    : {gray_row['Val Macro F1 (%)']:.2f}%")
print(f"  HOG YUV                     : {yuv_row['Val Macro F1 (%)']:.2f}%")
print(f"  HOG Gray + Color Hist (HSV) : {fusion_row['Val Macro F1 (%)']:.2f}%")
print()

if yuv_row["Val Macro F1 (%)"] < gray_row["Val Macro F1 (%)"]:
    print("-> Việc tăng số chiều với HOG YUV KHÔNG mang lại lợi ích tương xứng: "
          "F1 giảm so với HOG Gray dù nặng hơn nhiều lần và train chậm hơn "
          f"({yuv_row['Thời gian Train (s)']:.2f}s so với {gray_row['Thời gian Train (s)']:.2f}s). "
          "Có thể do nhiễu sắc độ ở kênh U/V và hiện tượng 'lời nguyền số chiều' (curse of dimensionality).")
else:
    print("-> HOG YUV cho F1 cao hơn HOG Gray trên dữ liệu thực tế của bạn — "
          "hãy đối chiếu lại với giả thuyết H2 trong báo cáo và giải thích nguyên nhân cụ thể.")

print()
best_row = df_results_m03.loc[df_results_m03["Val Macro F1 (%)"].idxmax()]
print("=" * 60)
print("BỘ ĐẶC TRƯNG CHIẾN THẮNG (dựa trên Val Macro F1 cao nhất):")
print(f"  Mã TN         : {best_row['Mã TN']}")
print(f"  Bộ đặc trưng  : {best_row['Bộ đặc trưng']}")
print(f"  Số chiều gốc  : {best_row['Số chiều (n_features)']}")
print(f"  Val Accuracy  : {best_row['Val Accuracy (%)']}%")
print(f"  Val Macro F1  : {best_row['Val Macro F1 (%)']}%")
print("=" * 60)

WINNER_FEATURE_NAME = best_row["Bộ đặc trưng"]
WINNER_N_FEATURES = int(best_row["Số chiều (n_features)"])


So sánh số chiều đặc trưng:
  HOG Gray                    : 900 chiều
  HOG YUV                     : 2700 chiều (gấp 3.0 lần HOG Gray)
  HOG Gray + Color Hist (HSV) : 1412 chiều

So sánh Val Macro F1:
  HOG Gray                    : 95.95%
  HOG YUV                     : 94.75%
  HOG Gray + Color Hist (HSV) : 95.19%

-> Việc tăng số chiều với HOG YUV KHÔNG mang lại lợi ích tương xứng: F1 giảm so với HOG Gray dù nặng hơn nhiều lần và train chậm hơn (21.30s so với 5.06s). Có thể do nhiễu sắc độ ở kênh U/V và hiện tượng 'lời nguyền số chiều' (curse of dimensionality).

BỘ ĐẶC TRƯNG CHIẾN THẮNG (dựa trên Val Macro F1 cao nhất):
  Mã TN         : TN2.1
  Bộ đặc trưng  : HOG Gray
  Số chiều gốc  : 900
  Val Accuracy  : 96.48%
  Val Macro F1  : 95.95%


**Nhận xét khoa học trong báo cáo**

- Trình bày rõ việc tăng số chiều lên gấp nhiều lần của `HOG YUV` có mang lại lợi ích thật sự hay chỉ làm chậm mô hình — dùng đúng số liệu Val F1 và thời gian train thực đo.
- Khẳng định (hoặc bác bỏ, nếu số liệu thực tế không ủng hộ) sự vượt trội của dung hợp đa thức `HOG Gray + Color Hist`, dựa trên bảng kết quả thật.
- Đối chiếu với Giả thuyết H2: kết quả có ủng hộ giả thuyết ban đầu không, và vì sao.

## 6. Bàn giao (Handover)

In [10]:
winner_data = feature_cache[WINNER_FEATURE_NAME]
X_train_final = winner_data["X_train"]
y_train_final = winner_data["y_train"]
X_val_final = winner_data["X_val"]
y_val_final = winner_data["y_val"]

os.makedirs("outputs", exist_ok=True)
np.save("outputs/M03_X_train.npy", X_train_final)
np.save("outputs/M03_y_train.npy", y_train_final)
np.save("outputs/M03_X_val.npy", X_val_final)
np.save("outputs/M03_y_val.npy", y_val_final)

handover_m03 = {
    "winner_feature_name": WINNER_FEATURE_NAME,
    "winner_n_features": WINNER_N_FEATURES,
    "winner_size": WINNER_SIZE,
    "winner_mode": WINNER_MODE,
    "hog_params": {k: (list(v) if isinstance(v, tuple) else v) for k, v in HOG_PARAMS.items()},
    "classes": CLASSES,
}
with open("outputs/M03_handover_config.json", "w", encoding="utf-8") as f:
    json.dump(handover_m03, f, ensure_ascii=False, indent=2)

print(f'"Bộ đặc trưng chiến thắng chính thức của dự án là: {WINNER_FEATURE_NAME} '
      f'với số chiều gốc là {WINNER_N_FEATURES}"')
print()
print("Đã lưu cho M04:")
print("  outputs/M03_X_train.npy, outputs/M03_y_train.npy")
print("  outputs/M03_X_val.npy,   outputs/M03_y_val.npy")
print("  outputs/M03_handover_config.json (mô tả bộ đặc trưng + tham số trích xuất)")


"Bộ đặc trưng chiến thắng chính thức của dự án là: HOG Gray với số chiều gốc là 900"

Đã lưu cho M04:
  outputs/M03_X_train.npy, outputs/M03_y_train.npy
  outputs/M03_X_val.npy,   outputs/M03_y_val.npy
  outputs/M03_handover_config.json (mô tả bộ đặc trưng + tham số trích xuất)


### Tổng kết M03

- [x] Đã chạy đủ 3 cấu hình đặc trưng trên đúng cấu hình tiền xử lý kế thừa từ M02.
- [x] Bảng kết quả lưu tại `outputs/M03_results_3_configs.csv`.
- [x] Ma trận đặc trưng chiến thắng (`X_train`, `X_val`, `y_train`, `y_val`) đã lưu dưới dạng `.npy` để dùng trực tiếp, không cần trích xuất lại.
- [x] Cấu hình chiến thắng lưu tại `outputs/M03_handover_config.json`.